# Task 4 – Distributed Computing

This notebook demonstrates Spark caching, persistence, repartitioning, and shows the cluster configuration that affects performance. It also records load, preprocessing, and model‑training times for later comparison.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import os, time, json

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Load data (reuse same path)
data_path = os.path.join(os.getcwd(), 'data', 'yellow_tripdata_2025-*.parquet')
df = spark.read.parquet(data_path)

# 1️⃣ Cache the DataFrame
df_cached = df.cache()
df_cached.count()  # materialise cache
print('DataFrame cached.')  # <--- Screenshot this confirmation

In [ ]:
# 2️⃣ Persist with MEMORY_AND_DISK
df_persisted = df.persist(StorageLevel.MEMORY_AND_DISK)
df_persisted.count()
print('DataFrame persisted with MEMORY_AND_DISK.')  # <--- Screenshot this confirmation

In [ ]:
# 3️⃣ Repartition to improve parallelism (200 partitions)
df_repart = df.repartition(200)
print(f'Number of partitions after repartition: {df_repart.rdd.getNumPartitions()}')  # <--- Screenshot this output

## Spark Configuration Overview

In [ ]:
conf_dict = dict(spark.sparkContext.getConf().getAll())
for k, v in conf_dict.items():
    print(f'{k}: {v}')
# Render as a markdown table for the report
import pandas as pd
conf_df = pd.DataFrame(list(conf_dict.items()), columns=['Config', 'Value'])
display(conf_df)  # <--- Screenshot this table

## Timing Measurements (Load → Preprocess → Train)
We reuse the preprocessing pipeline from Task 2 and train a Linear Regression model as a representative workload.

In [ ]:
# Load time
t0 = time.time()
df = spark.read.parquet(data_path)
load_time = time.time() - t0
print(f'Load time: {load_time:.2f}s')  # <--- Screenshot this output

In [ ]:
# Preprocess time (reuse pipeline from Task2)
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Pipeline
cat_cols = ['vendor_id', 'store_and_fwd_flag']
indexers = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_vec') for c in cat_cols]
num_cols = ['passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude',
                    'dropoff_longitude', 'dropoff_latitude', 'trip_duration',
                    'pickup_hour', 'pickup_day', 'pickup_month']
assembler = VectorAssembler(inputCols=num_cols + [c+'_vec' for c in cat_cols], outputCol='features_assembled')
scaler = StandardScaler(inputCol='features_assembled', outputCol='features', withMean=True, withStd=True)
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
t0 = time.time()
pipeline_model = pipeline.fit(df)
prep_time = time.time() - t0
print(f'Preprocessing time: {prep_time:.2f}s')  # <--- Screenshot this output

In [ ]:
# Model training time (Linear Regression)
from pyspark.ml.regression import LinearRegression
model_df = pipeline_model.transform(df).select('features', 'fare_amount')
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
lr = LinearRegression(featuresCol='features', labelCol='fare_amount')
t0 = time.time()
lr_model = lr.fit(train_df)
train_time = time.time() - t0
print(f'Linear Regression training time: {train_time:.2f}s')  # <--- Screenshot this output

## Spark UI Guidance (Screenshots)
Below are the tabs you should capture from the Spark UI while the notebook runs:
- **Jobs**: Shows overall job progress and duration.
- **Stages**: Breakdown of each stage, task distribution, and shuffle stats.
- **Executors**: Memory/CPU usage per executor.
- **Storage**: Cached/persisted DataFrames and their memory footprint.
*Take a screenshot of each tab and embed it in the final report.*

## Caching & Repartitioning Justifications
> **NOTE** Caching is justified because the same DataFrame is accessed multiple times (loading, preprocessing, model training).
> **NOTE** Repartitioning to 200 partitions aligns with `spark.sql.shuffle.partitions` and balances parallelism across the available cores.